# REST API

Every method at `POST /methods/<name>`, with a bearer token. Here the app runs
in-process, so nothing listens on a port; `uv run amoorion api` serves it for real.

In [1]:
import os

# The API's token, as the vault would hold it.
# The API's token comes from the vault; made up, in this sandbox's variables.
os.environ["AMOORION_SECRET_API_TOKEN"] = "api-example-token"

In [2]:
# Calls the application directly, so no port is opened.
from fastapi.testclient import TestClient

from amoorion import Amoorion

# `.api()` is an ordinary FastAPI application, for any ASGI server.
client = TestClient(Amoorion().api())

# Reading the list of methods needs no token.
client.get("/").json()["methods"]

['accounts',
 'compose',
 'delete',
 'inbox',
 'notify',
 'plugins',
 'publish',
 'read',
 'reply',
 'search',
 'simulate',
 'skill',
 'snapshot',
 'wiki',
 'workflow']

No token, no call:

In [3]:
# Without a token: 401, the API refuses.
client.post(
    "/methods/publish",
    json={
        "text": "x",
        "to": ["dryrun"],
    },
).status_code

401

In [4]:
# The same token as the vault's.
auth = {"Authorization": "Bearer api-example-token"}

# One route per method, the JSON being its parameters.
client.post(
    "/methods/publish",
    json={"text": "over HTTP", "to": ["dryrun"]},
    headers=auth,
).json()[0]["ok"]

True

In [5]:
# Everything goes to one place, however it got posted.
client.post(
    "/methods/search",
    json={"text": "http"},
    headers=auth,
).json()

[]

The schema of it all, for generated clients and for tools like Dify:

In [6]:
# Described by OpenAPI, for generating clients.
sorted(client.get("/openapi.json").json()["paths"])[:4]

['/', '/hooks/{name}', '/methods/accounts', '/methods/compose']